In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from numpy import cos,pi,sqrt
from astropy.table import Table
from astropy.table import unique
import os
import ctadata
from astropy.io import ascii

In [2]:
gheff='gheff0.9'


In [3]:
run_catalog=ascii.read('LST_source_catalog.ecsv')
run_catalog.columns

good_runs=np.load('good_runs_Zd_80.0.npy')
1874 in good_runs

True

In [4]:
dates_DL3=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/')

In [ ]:
runs=[]
flist=[]
texpos=[]
ra_pnts=[]
dec_pnts=[]
src_names=[]
alt_pnts=[]
az_pnts=[]
ra_objs=[]
dec_objs=[]
nsb_tunings=[]
dates=[]
point_fulls=[]
tstarts=[] 
counter=0
for i in range(len(run_catalog)):
    run=run_catalog[i]['Run ID']
    if(run
    src=run_catalog[i]['Source name']
    date=run_catalog[i]['Date directory'].replace('-','')
    if(date in dates_DL3)and(run in good_runs): 
        vers=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date)
        for ver in vers:
            if(ver[0]=='v'):
                tailcuts=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver)
                for tailcut in (tailcuts):
                    nsbs=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut)
                    for nsb in nsbs:
                        versions=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb))
                        for version in versions:
                            tags=ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std')
                            for tag in tags:
                                src_dependences=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag))
                                for src_dep in src_dependences:
                                    point_or_full=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep))
                                    for p_f in point_or_full:
                                        wobbles=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f))
                                        for wob in wobbles:
                                            gheffs=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob))        
                                            for gh in gheffs:
                                                if(gheff in gh):
                                                    irfs=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh))
                                                    for irf in irfs:
                                                        files=(ctadata.list_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf))
                                                        if(run<10000):
                                                            fname='dl3_LST-1.Run0'+str(run)+'.fits'
                                                        else:
                                                            fname='dl3_LST-1.Run'+str(run)+'.fits'
                                                        if(fname in files):
                                                            runs.append(run)
                                                            flist.append('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                            f=('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                            counter+=1
                                                            ctadata.fetch_and_save_file_or_dir('/pnfs/cta.cscs.ch/lst/DL3/'+date+'/'+ver+'/'+tailcut+'/'+nsb+'/'+version+'/std/'+tag+'/'+src_dep+'/'+p_f+'/'+wob+'/'+gh+'/'+irf+'/'+fname)
                                                            hdul=fits.open(fname)
                                                            header=hdul['EVENTS'].header
                                                            texpos.append(header['LIVETIME'])
                                                            src_names.append(src)
                                                            alt_pnts.append(header['ALT_PNT'])
                                                            az_pnts.append(header['AZ_PNT'])
                                                            ra_objs.append(header['RA_OBJ'])
                                                            dec_objs.append(header['DEC_OBJ'])
                                                            mjdref=header['MJDREFI']
                                                            nsb_tunings.append(nsb[-4:])
                                                            tstarts.append(mjdref+header['TSTART']/86400)
                                                            dates.append(date)
                                                            point_fulls.append(p_f)
                                                            print(counter,date,fname,src_names[-1],texpos[-1],p_f)
                                                            ra_pnts.append(header['RA_PNT'])
                                                            dec_pnts.append(header['DEC_PNT'])

                                                            if(counter%100==0):
                                                                cat = Table()
                                                                cat['Run ID'] = runs
                                                                cat['RA_PNT'] = ra_pnts
                                                                cat['DEC_PNT']=dec_pnts
                                                                cat['Exposure[s]']=texpos
                                                                cat['ALT_PNT']=alt_pnts
                                                                cat['AZ_PNT']=az_pnts
                                                                cat['TSTART[MJD]']=tstarts
                                                                cat['SRC_NAME']=src_names
                                                                cat['NSB_TUNING']=nsb_tunings
                                                                cat['POINT/FULL_ENCLOSURE']=point_fulls
                                                                cat['FILENAME']=flist
                                                                ascii.write(cat, 'run_catalog_dark_nights'+str(counter)+'.dat', overwrite=True)
    
                                                            os.remove(fname)
    #else:
    #    print(date+' not in dCache')


1 20200127 dl3_LST-1.Run01874.fits Crab 1123.563894333783 point
2 20200127 dl3_LST-1.Run01880.fits Crab 1107.92094744185 point
3 20200128 dl3_LST-1.Run01888.fits Crab 217.44090590173087 point
4 20200131 dl3_LST-1.Run01918.fits Mrk421 1692.4957245962066 point
5 20200131 dl3_LST-1.Run01919.fits Mrk421 785.0888872987113 point
6 20200215 dl3_LST-1.Run01991.fits Crab 1153.8519179691548 point
7 20200215 dl3_LST-1.Run01992.fits OffCrab 521.2365496902214 full-enclosure
8 20200215 dl3_LST-1.Run01992.fits OffCrab 521.2365496902214 point
9 20200218 dl3_LST-1.Run02009.fits OffCrab 1146.5676856951807 full-enclosure
10 20200218 dl3_LST-1.Run02009.fits OffCrab 1146.5676856951807 point
11 20200227 dl3_LST-1.Run02033.fits OffCrab 1102.8755417079628 full-enclosure
12 20200227 dl3_LST-1.Run02033.fits OffCrab 1102.8755417079628 point
13 20200713 dl3_LST-1.Run02205.fits 1ES1959+650 708.5775941294355 point
14 20200713 dl3_LST-1.Run02206.fits 1ES1959+650 900.4867699902567 point
15 20200713 dl3_LST-1.Run02207

In [ ]:
cat = Table()
cat['Run ID'] = runs
cat['RA_PNT'] = ra_pnts
cat['DEC_PNT']=dec_pnts
cat['Exposure[s]']=texpos
cat['ALT_PNT']=alt_pnts
cat['AZ_PNT']=az_pnts
cat['TSTART[MJD]']=tstarts
cat['SRC_NAME']=src_names
cat['NSB_TUNING']=nsb_tunings
cat['POINT/FULL_ENCLOSURE']=point_fulls
cat['FILENAME']=flist
ascii.write(cat, 'run_catalog_dark_nights.dat', overwrite=True)

In [ ]:
ascii.read('run_catalog_dark_nights.dat')